# Precision at fixed candidate budgets, ranked by chromatin density, containment-gated click-centred seeds (K = 10, 20, 30, 50)

**Question.** Replicates `gated_seed_precision_at_k.ipynb` exactly through NMS -- same 14
ROIs, same one click per ROI, same search, same deep-floor extraction, same NMS -- and changes
only the ranking step. Instead of the single `tm_score` (`TM_CCOEFF`) arm, six arms rank the
same post-NMS candidate pool: `tm_score` (D5's production ranker, included as the baseline) and
five chromatin-density axes named in `DECISIONS.md` D5's "what would change my mind" clause --
`chromatin_od` (`od51`), `od31`, `od_falloff`, `mask_od_mean`, `od_contrast`.

**What this variant is.** A copy of
`precision_at_k_budgets_14roi_chromatin/precision_at_k_budgets_14roi_chromatin.ipynb` with exactly
one change: the seed's template size comes from `seed_selection.tightened_base_size` -- the Otsu
component the click's own pixel lands inside -- rather than that notebook's inline `largest_cc_box`,
which sized on the window's largest component whether or not the click was in it. The template is
still built centred on the raw click, and all six ranking arms are untouched. That pair --
containment gate and size correction kept, position correction dropped -- is what `DECISIONS.md`
D8's "Amendment, 2026-09-09 -- recentring is reversed" settles on. Gate 0's reference run moves with
it, from the ungated `precision_at_k_budgets_14roi.ipynb` to this family's own
`gated_seed_precision_at_k.ipynb`, so the pool this notebook re-ranks is the one built from the same
seeds.

**Scope, fixed for this run:**
- 14 ROIs, `images/extra_valid` (2 per tumour domain x 7 domains).
- 1 seed per ROI, `seed_index = 0` -- identical RNG stream to the reference notebook, so the
  seed draw, `base_size` and candidate pool reproduce byte-for-byte (checked below, Gate 0).
- **Seed construction: click-centred, containment-gated** (`seed_selection.tightened_base_size`),
  per `DECISIONS.md` D8's "Amendment, 2026-09-09 -- recentring is reversed". The template's *size*
  comes from the Otsu component the click's own pixel sits inside; its *centre* stays the raw
  click. A click outside its component is refused as a seed and redrawn on the same RNG stream.
  This replaces the source notebooks' inline `largest_cc_box`, which took the window's largest
  component with no containment check, and is **not** the recentred `tightened_template_box`
  variant the same amendment reverses.
- Today's decided defaults (`DECISIONS.md`): `TM_CCOEFF` (D1), no `tissue_mask` (D2),
  `hematoxylin_od` unclipped (D3), NMS radius = match radius = 7.5 um (D7, enforced via
  `invariants.check_nms_radius`).
- No z-threshold sweep, no seed sweep. Candidates are extracted once per ROI at the permissive
  deep floor (`z = -1.5`), NMS'd at 7.5 um -- exactly as in the reference notebook -- and then
  `compare.evaluate_arms` ranks and re-matches that one pool independently for each of the six
  arms, truncating to K = 10/20/30/50 per arm.
- This is **not** a seed sweep and does not settle D5 (which requires 5 seeds, paired delta
  clustered at the ROI). It reports single-seed precision@K per arm, which is what this
  notebook's own metric can say.

In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils import invariants as inv
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Configuration. Identical to `gated_seed_precision_at_k.ipynb` through NMS -- D1/D2/D3/D7
# per DECISIONS.md. The one change is the ranking step: instead of `tm_score` alone (D5's
# production ranker), six arms rank the SAME post-NMS pool -- tm_score plus the five
# chromatin-density axes D5's "what would change my mind" clause names (chromatin_od/od51,
# od31, od_falloff, mask_od_mean, od_contrast). No z-sweep, no seed-sweep: single deep-floor
# extraction, single NMS, single click per ROI -- same scope as the reference notebook.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0                       # one seed per ROI

CHANNEL = 'hematoxylin_od'           # D3 -- unclipped optical density
METHOD = cv2.TM_CCOEFF               # D1 -- unnormalized, contrast-sensitive
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5                  # near-unfiltered extraction floor (repo convention)
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM   # 7.5 um -- D7: NMS radius == match radius
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2         # 36
OTSU_WINDOW = tm.BASE_SIZE           # 51

# Chromatin-axis windows (midog_utils.chromatin.chromatin_density, tp_fp_feature_extract's
# shape_features). od31/od51/od81 share `frac=0.10` (chromatin_density's default: mean of the
# darkest 10% of the window); od_ctx uses the darkest half of a wider neighbourhood. One pad
# covers every window used below: 60 px is exactly `read_padded_patch`'s requirement for the
# 121 px od_ctx window (half = window // 2) and more than enough for every smaller one.
OD_WINDOWS = (31, 51, 81)
CTX_WINDOW, CTX_FRAC = 121, 0.50
SHAPE_WINDOW = tm.BASE_SIZE          # 51 -- tp_fp_feature_extract.SHAPE_WINDOW
OD_PAD = CTX_WINDOW // 2             # 60 -- exactly covers od_ctx (121px), the largest window

# rank_key -> the pool column each arm sorts on, descending, NaN last (compare._rank).
# 'tm_score' is the D5 production ranker, included as the baseline every chromatin axis is
# measured against, per D5's own rule that no axis may be judged without that comparison.
AXES = {
    'tm_score':     'score',
    'chromatin_od': 'od51',
    'od31':         'od31',
    'od_falloff':   'od_falloff',
    'mask_od_mean': 'mask_od_mean',
    'od_contrast':  'od_contrast',
}

BUDGETS = (10, 20, 30, 50)           # the fixed candidate-list lengths under test

OUT_PER_ROI = '../results/precision_at_k_14roi_gatedseed_chromatin_per_roi.csv'
OUT_BY_DOMAIN = '../results/precision_at_k_14roi_gatedseed_chromatin_by_domain.csv'
OUT_RAW = '../results/precision_at_k_14roi_gatedseed_chromatin_raw.csv'
OUT_VERIF = '../results/precision_at_k_14roi_gatedseed_chromatin_verification.csv'

# The tm_score-only reference run this notebook must reproduce through NMS, used only for
# the Gate 0 reproduction check below -- never re-read after that. This family's own base run
# (gated_seed_precision_at_k.ipynb), not the ungated precision_at_k_budgets_14roi.ipynb the
# source version of this notebook gated against: Gate 0 is only meaningful against a run that
# shares this one's seed construction.
REFERENCE_PER_ROI = '../results/precision_at_k_14roi_gatedseed_per_roi.csv'

print(f'budgets {BUDGETS} x 14 ROIs x 1 click (seed_index={SEED_INDEX}) x {len(AXES)} arms')
print(f'arms: {list(AXES)}')
print(f'NMS radius = match radius = {NMS_RADIUS_UM} um (D7)')

budgets (10, 20, 30, 50) x 14 ROIs x 1 click (seed_index=0) x 6 arms
arms: ['tm_score', 'chromatin_od', 'od31', 'od_falloff', 'mask_od_mean', 'od_contrast']
NMS radius = match radius = 7.5 um (D7)


## The pipeline

Copied from `precision_at_k_budgets_14roi_chromatin/precision_at_k_budgets_14roi_chromatin.ipynb`,
which copied it from `precision_at_k_budgets_14roi.ipynb`, which copied it from
`find_and_suppress_high_threshold_precision.ipynb`, which copied it from
`recall_workload_ledger.py` -- the seed draw and the NMS+self-hit helper are kept inline rather than
promoted to `midog_utils`, for the same reason those notebooks give. `shape_features` is added here,
copied inline from `tp_fp_feature_extract.py` -- the gate-free largest-Otsu-component intensity mean
D6 found beats the gated version `midog_utils/chromatin.py`'s own docstring originally rejected it
in favour of.

**Seed sizing is the one change in this notebook, and the reason it exists.** The source
notebooks size the seed's template with an inline `largest_cc_box`: the *largest* Otsu component
in the click's 51 px window, taken whether or not the click itself lands inside it. This one calls
`seed_selection.tightened_base_size` instead, which routes the seed through
`seed_selection.tighten_box_otsu`'s containment gate -- the component measured is the one the
click's own rounded pixel is foreground of (`center_tolerance=0`), and it must still clear the same
`min_area=50` / `max_area_frac=0.85` / `min_solidity=0.5` checks the inline helper applied. A click
that is not inside any accepted component returns `None`, which `draw_seed_with_retry` treats
exactly as it treated a failed gate before: drop that candidate, redraw on the same RNG stream,
count it in `n_retries`. `tightened_base_size` reads its own patch and returns an odd `base_size`
directly, so the manual `read_padded_patch`, the bbox unpacking and the local `_odd_local` are all
gone with the helper.

**Size only -- the click stays the centre.** `seed_xy` is the raw click, the template patch is read
there, and self-hit removal, the seed-annulus check and every ground-truth match are all referenced
to it, exactly as in the source notebooks. `DECISIONS.md` D8's "Amendment, 2026-09-09 -- recentring
is reversed" keeps the containment gate and the size correction and rejects the position correction
outright ("size still comes from the accepted component, position reverts to the click"), so
`seed_selection.tightened_template_box` -- the recentred variant D8 originally decided on -- is
deliberately not called here, and no recentred point exists anywhere in this notebook.

**The gate is a seed rule, not a candidate rule.** `shape_features` below still takes the largest
Otsu component under a candidate with no containment check and no accept/reject gates at all -- that
is deliberate and unchanged: D6 measured the gate-free version as the better `mask_od_mean`, and it
scores candidates rather than deciding what a template is built from. Only seed construction is
gated here.

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    '''Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.'''
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, seed_xy):
    '''NMS at `radius`, then drop the seed's own self-correlation. Returns (centers, scores).'''
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - seed_xy[0], c[:, 1] - seed_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def shape_features(chan, cx, cy, window=SHAPE_WINDOW):
    '''Largest Otsu component under a point, measured with no accept/reject gate.

    Copied from `tp_fp_feature_extract.py` -- per-patch min-max to uint8, then binary Otsu, then
    the largest connectivity-2 component, with every gate removed and no containment check, so
    `mask_od_mean` is defined for every candidate Otsu finds any foreground in (D6: gate-free
    beats the gated version on AUC, look-alike contrast and read_95). Unrelated to the seed's own
    sizing step above, which is containment-gated by `seed_selection.tightened_base_size`; this
    one ranks candidates, it does not decide what a template is built from. Returns
    None only when the window is unreadable or Otsu finds no foreground at all -- the caller
    ranks that `nan`, last, rather than dropping the candidate.
    '''
    patch = tm.read_padded_patch(chan, cx, cy, window)
    if patch is None:
        return None
    u8 = cv2.normalize(patch.astype(np.float32), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, binary = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    regions = regionprops(label(binary, connectivity=2), intensity_image=patch)
    if not regions:
        return None
    r = max(regions, key=lambda x: x.area)
    return {'mask_od_mean': float(r.intensity_mean)}


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## The run

Per ROI: one `matchTemplate` pass (the expensive step, ~15 s), one extraction at the deep
floor, one NMS at 7.5 um -- identical to the reference notebook. Then, on the surviving pool:
one border-replicate pad at 60 px, four `chromatin_density` windows (31/51/81 px darkest-10%,
121 px darkest-50%) and one gate-free Otsu pass per candidate (`shape_features`), from which
`od_contrast = od51 - od_ctx` and `od_falloff = od31 - od81` are derived. Six `compare.Arm`s --
`tm_score` plus five chromatin axes -- then rank and re-match this one pool independently
(`compare.evaluate_arms`, `midog_utils/compare.py`), so precision@K for every axis comes from
the *same* candidates, differing only in sort order.

In [3]:
def run_roi(fn, image_id, domain, anns):
    t0 = time.time()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb
    gc.collect()

    # --- the click -------------------------------------------------------------------
    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    # Containment-gated, size-only, click-centred -- DECISIONS.md D8's 2026-09-09 amendment.
    # `tightened_base_size` reads its own OTSU_WINDOW patch, runs `tighten_box_otsu` (the
    # click's own pixel must be foreground of a component clearing min_area/max_area_frac/
    # min_solidity), and returns that component's odd longer side. None -> this click is not
    # inside any accepted component; draw_seed_with_retry drops it and redraws on the same
    # RNG stream. The template centre is untouched: seed_xy below is still the raw click.
    def _check(row):
        return ss.tightened_base_size(gray_inv, float(row['cx']), float(row['cy']),
                                       otsu_window=OTSU_WINDOW)

    seed, base_size, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_xy = (float(seed['cx']), float(seed['cy']))
    seed_ann_id = int(seed['ann_id'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    del gray_inv
    gc.collect()

    # --- one match, one deep-floor extraction, one NMS -- no z-sweep --------------------
    patch = tm.read_padded_patch(hem, *seed_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    del hem_p, fused_p, valid_p
    gc.collect()

    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    n_peaks = len(centers)
    assert n_peaks < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    c, s = suppress(centers, scores, nms_radius, seed_xy)
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})

    # Seed annulus: at NMS radius == match radius, NMS itself empties this before self-hit
    # removal ever runs (the self-correlation is the map's global maximum, kept first, and
    # suppresses everything within one NMS radius) -- so this must count zero here.
    d_seed = np.hypot(pool['cx'] - seed_xy[0], pool['cy'] - seed_xy[1])
    n_near_seed = int((d_seed <= match_radius).sum())

    del fused, valid, templates, patch, centers, scores, c, s
    gc.collect()

    # --- chromatin axes on the NMS survivors -- one pad covers every window used below ---
    hem_pad = cv2.copyMakeBorder(hem, OD_PAD, OD_PAD, OD_PAD, OD_PAD, cv2.BORDER_REPLICATE)
    px = pool['cx'].to_numpy() + OD_PAD
    py = pool['cy'].to_numpy() + OD_PAD
    for w in OD_WINDOWS:
        pool[f'od{w}'] = [cm.chromatin_density(hem_pad, x, y, window=w) for x, y in zip(px, py)]
    pool['od_ctx'] = [cm.chromatin_density(hem_pad, x, y, window=CTX_WINDOW, frac=CTX_FRAC)
                      for x, y in zip(px, py)]
    pool['od_contrast'] = pool['od51'] - pool['od_ctx']
    pool['od_falloff'] = pool['od31'] - pool['od81']
    n_od_nan = int(pool[[f'od{w}' for w in OD_WINDOWS] + ['od_ctx']].isna().sum().sum())
    assert n_od_nan == 0, f'{fn}: {n_od_nan} NaN od survived the {OD_PAD} px pad'

    shape_rows = [shape_features(hem_pad, x, y, window=SHAPE_WINDOW) for x, y in zip(px, py)]
    shape_ok = np.array([r is not None for r in shape_rows])
    pool['mask_od_mean'] = [r['mask_od_mean'] if r is not None else np.nan for r in shape_rows]
    shape_fail_rate = float(1 - shape_ok.mean()) if len(shape_ok) else float('nan')

    del hem, hem_pad
    gc.collect()

    # --- six arms, one pool, independent rank + re-match per arm -------------------------
    # coverage_key=fn: all six arms share the same candidate positions (same `pool`, just
    # resorted), and coverage_fraction depends only on those positions, not rank order or
    # rank_key (evaluate.coverage_fraction takes only cx, cy) -- so evaluate_arms's own
    # coverage_cache computes it once per ROI on the first arm and reuses it for the rest.
    arms = [
        cp.Arm(name, (lambda d=pool: d), rank_key=key, seeded=True, z=DEEP_FLOOR_Z,
               z_dependent=True, nms_radius=nms_radius, caps=(MAX_PEAKS,), coverage_key=fn)
        for name, key in AXES.items()
    ]
    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id,
               seed_ann_id=seed_ann_id, base_size=base_size,
               map_median=round(float(med), 5), mad_scale=round(float(mad), 5), mpp=mpp)
    checks = []
    out = cp.evaluate_arms(arms, gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                            budgets=BUDGETS, context=ctx, checks=checks)

    checks.append(dict(check='seed_annulus_empty', label=fn, n_near_seed=n_near_seed,
                       match_radius_px=round(match_radius, 3), passed=bool(n_near_seed == 0)))

    meta = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                n_retries=n_retries, contested_seed_tier=bool(flagged), base_size=base_size,
                mpp=mpp, roi_h=int(H), roi_w=int(W), pad_px=PAD,
                match_radius_px=match_radius, nms_radius_px=nms_radius,
                map_median=float(med), mad_scale=float(mad), n_gt_mitotic=n_gt,
                n_detections=len(pool), n_near_seed_annulus=n_near_seed,
                shape_fail_rate=round(shape_fail_rate, 5),
                t_total_s=round(time.time() - t0, 1))
    print(f"[{fn}] {domain:32s} base={base_size:2d} n_detections={len(pool):6d} "
          f"n_gt={n_gt:3d} shape_fail={shape_fail_rate:.3%} [{meta['t_total_s']:.0f}s]", flush=True)
    return out, pd.DataFrame(checks), meta

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
missing = sorted(set(files) - set(meta_ix.index))
assert not missing, f'.tiff on disk absent from the annotation DB: {missing}'
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

t_run = time.time()
out_frames, check_frames, roi_meta = [], [], []
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    out, checks, m = run_roi(fn, image_id, domain, annotations)
    out_frames.append(out)
    check_frames.append(checks)
    roi_meta.append(m)
    gc.collect()

RAW = pd.concat(out_frames, ignore_index=True)
VERIF = pd.concat(check_frames, ignore_index=True)
ROI = pd.DataFrame(roi_meta).set_index('file_name')

print(f'\n{len(files)} ROIs x {len(BUDGETS)} budgets x {len(AXES)} arms = {len(RAW)} rows in {time.time() - t_run:.0f}s')
ROI[['tumor_type', 'seed_ann_id', 'base_size', 'n_gt_mitotic', 'n_detections',
     'match_radius_px', 'nms_radius_px', 'map_median', 'mad_scale', 'shape_fail_rate',
     't_total_s']].round(3)

[013.tiff] human breast cancer              base=31 n_detections= 17200 n_gt= 17 shape_fail=0.000% [67s]


[094.tiff] human breast cancer              base=25 n_detections= 18410 n_gt= 81 shape_fail=0.000% [32s]


[201.tiff] canine lung cancer               base=51 n_detections= 15848 n_gt= 17 shape_fail=0.000% [27s]


[233.tiff] canine lung cancer               base=25 n_detections= 17614 n_gt= 17 shape_fail=0.000% [42s]


[245.tiff] canine lymphosarcoma             base=47 n_detections= 17552 n_gt= 89 shape_fail=0.000% [63s]


...

[460.tiff] canine soft tissue sarcoma       base=47 n_detections= 15339 n_gt= 35 shape_fail=0.000% [18s]


[529.tiff] human melanoma                   base=37 n_detections= 15862 n_gt= 19 shape_fail=0.000% [23s]


[548.tiff] human melanoma                   base=29 n_detections= 17413 n_gt=238 shape_fail=0.000% [18s]



14 ROIs x 4 budgets x 6 arms = 336 rows in 429s


,tumor_type,seed_ann_id,base_size,n_gt_mitotic,n_detections,match_radius_px,nms_radius_px,map_median,mad_scale,shape_fail_rate,t_total_s
file_name,,,,,,,,,,,
013.tiff,human breast cancer,254,31,17,17200,33.139,33.139,-0.005,0.128,0.0,67.2
094.tiff,human breast cancer,2512,25,81,18410,32.630,32.630,-0.003,0.051,0.0,31.8
201.tiff,canine lung cancer,4457,51,17,15848,30.222,30.222,-0.028,0.612,0.0,27.1
233.tiff,canine lung cancer,5761,25,17,17614,30.222,30.222,-0.016,0.144,0.0,41.8
245.tiff,canine lymphosarcoma,6274,47,89,17552,30.222,30.222,-0.007,0.214,0.0,63.1
246.tiff,canine lymphosarcoma,6548,41,115,17763,30.222,30.222,-0.006,0.171,0.0,36.6
300.tiff,canine cutaneous mast cell tumor,14581,45,180,17940,29.609,29.609,-0.057,0.371,0.0,25.8
301.tiff,canine cutaneous mast cell tumor,14969,41,217,17963,29.609,29.609,-0.022,0.204,0.0,19.4
402.tiff,human neuroendocrine tumor,20254,29,104,17337,33.139,33.139,-0.056,0.921,0.0,18.6


## Checks before any table

Four things must hold: the three from the reference notebook (deep-floor pool clears the
largest budget; `budget_delivered == budget` at every K; no surviving candidate lies within
one match radius of the removed seed), plus one new to this notebook -- **Gate 0**, that this
run's `tm_score` arm reproduces the `tp_at_budget` of this family's base run
(`gated_seed_precision_at_k.ipynb`, read via `REFERENCE_PER_ROI`) at every (ROI, K), not merely
a matching candidate *count*. Pool size alone would wave through a coincidental
same-size, different-content pool; matching `tp_at_budget` depends on the exact candidates and
the exact greedy match against ground truth, which is what actually establishes this is the
same pool the reference notebook scored -- so any precision difference in Table A/B below is
attributable only to the ranking step, not to a drifted search or NMS.

`invariants.check_nms_radius` (D7) and `invariants.check_no_cap` were already asserted per
(ROI, arm) inside `evaluate_arms`, above, and are already sitting in `VERIF`.

In [5]:
REF = pd.read_csv(REFERENCE_PER_ROI).set_index('file_name')
common = ROI.index.intersection(REF.index)
assert len(common) == 14, f'expected 14 ROIs in common with {REFERENCE_PER_ROI}, found {len(common)}'

# Gate 0: pool size matching alone (n_detections) would wave through a coincidental
# same-size, different-content pool. Matching tp_at_budget at every (ROI, K) for the
# tm_score arm depends on the exact candidates, their exact (cx, cy), and the exact greedy
# match against ground truth -- that is what actually establishes "the same pool", so it is
# checked directly rather than assumed from a count.
tm_tp = (RAW[RAW['arm'] == 'tm_score']
         .pivot_table(index='file_name', columns='budget', values='tp_at_budget'))
ref_tp = REF[[f'tp_at_{k}' for k in BUDGETS]].copy()
ref_tp.columns = list(BUDGETS)

size_mismatch = ROI.loc[common, 'n_detections'].astype(int) != REF.loc[common, 'n_detections'].astype(int)
tp_mismatch = tm_tp.loc[common].astype(int) != ref_tp.loc[common].astype(int)
gate0_passed = bool(not size_mismatch.any() and not tp_mismatch.any().any())
if not gate0_passed:
    print('n_detections mismatches:')
    print(ROI.loc[common[size_mismatch], ['n_detections']].join(
        REF.loc[common[size_mismatch], ['n_detections']], lsuffix='_this', rsuffix='_reference'))
    print('tp_at_budget mismatches (tm_score arm):')
    print(tp_mismatch[tp_mismatch.any(axis=1)])
assert gate0_passed, 'tm_score arm did not reproduce the reference run -- see printout above'

assert (ROI['n_detections'] >= max(BUDGETS)).all(), \
    'a deep-floor pool did not clear the largest budget -- see ROI[\'n_detections\']'
assert (RAW['budget_delivered'] == RAW['budget']).all(), \
    'a budget row was starved -- the deep floor did not have enough survivors somewhere'
assert (ROI['n_near_seed_annulus'] == 0).all(), \
    'a candidate survived inside the removed seed\'s match radius -- see n_near_seed_annulus'

VERIF = pd.concat([VERIF, pd.DataFrame([
    dict(check='reproduces_reference_tp_at_budget', label='ALL', passed=gate0_passed),
    dict(check='deep_pool_covers_max_budget', label='ALL',
         passed=bool((ROI['n_detections'] >= max(BUDGETS)).all())),
    dict(check='no_starvation_any_budget', label='ALL',
         passed=bool((RAW['budget_delivered'] == RAW['budget']).all())),
])], ignore_index=True)
VERIF.to_csv(OUT_VERIF, index=False)
print(f'{len(VERIF)} verification records -> {OUT_VERIF}')
print(f"  passed: {int(VERIF['passed'].sum())} / {len(VERIF)}")
assert VERIF['passed'].all(), 'a check failed -- read the verification CSV before any table below'

185 verification records -> ../results/precision_at_k_14roi_gatedseed_chromatin_verification.csv
  passed: 185 / 185


## Table A -- precision per ROI, per arm, at each budget

One row per (ROI, arm) -- 14 ROIs x 6 arms = 84 rows, sorted by domain then arm. `n_gt_mitotic`
and `n_detections` are descriptive context and identical across arms within a ROI (same pool).
No `z_at_rank_K` diagnostic here: unlike `tm_score`, the chromatin axes are optical-density
values (D3: comparable within one ROI's ranking, not between ROIs or against a z-scale), so a
single "value at rank K" column would not mean the same thing across arms.

In [6]:
RAW['precision_at_budget'] = RAW['tp_at_budget'] / RAW['budget_delivered']

order = ROI[['tumor_type']].reset_index().sort_values(['tumor_type', 'file_name'])

rows_a = []
for _, o in order.iterrows():
    fn = o['file_name']
    for arm in AXES:
        r = dict(file_name=fn, domain=o['tumor_type'], arm=arm,
                 n_gt_mitotic=int(ROI.loc[fn, 'n_gt_mitotic']),
                 n_detections=int(ROI.loc[fn, 'n_detections']))
        sub = RAW[(RAW['file_name'] == fn) & (RAW['arm'] == arm)].set_index('budget')
        for k in BUDGETS:
            r[f'budget_delivered_{k}'] = int(sub.loc[k, 'budget_delivered'])
            r[f'tp_at_{k}'] = int(sub.loc[k, 'tp_at_budget'])
            r[f'precision_at_{k}'] = round(float(sub.loc[k, 'precision_at_budget']), 4)
        rows_a.append(r)

TABLE_A = pd.DataFrame(rows_a)
TABLE_A.to_csv(OUT_PER_ROI, index=False)
print(f'-> {OUT_PER_ROI}  ({len(TABLE_A)} rows = 14 ROIs x {len(AXES)} arms)')
TABLE_A

-> ../results/precision_at_k_14roi_gatedseed_chromatin_per_roi.csv  (84 rows = 14 ROIs x 6 arms)


,file_name,domain,arm,n_gt_mitotic,n_detections,budget_delivered_10,tp_at_10,precision_at_10,budget_delivered_20,tp_at_20,precision_at_20,budget_delivered_30,tp_at_30,precision_at_30,budget_delivered_50,tp_at_50,precision_at_50
0,300.tiff,canine cutaneous mast cell tumor,tm_score,180,17940,10,7,0.7,20,14,0.70,30,20,0.6667,50,32,0.64
1,300.tiff,canine cutaneous mast cell tumor,chromatin_od,180,17940,10,9,0.9,20,18,0.90,30,27,0.9000,50,43,0.86
2,300.tiff,canine cutaneous mast cell tumor,od31,180,17940,10,10,1.0,20,19,0.95,30,28,0.9333,50,46,0.92
3,300.tiff,canine cutaneous mast cell tumor,od_falloff,180,17940,10,10,1.0,20,19,0.95,30,27,0.9000,50,44,0.88
4,300.tiff,canine cutaneous mast cell tumor,mask_od_mean,180,17940,10,9,0.9,20,17,0.85,30,26,0.8667,50,42,0.84
5,300.tiff,canine cutaneous mast cell tumor,od_contrast,180,17940,10,9,0.9,20,18,0.90,30,26,0.8667,50,44,0.88
6,301.tiff,canine cutaneous mast cell tumor,tm_score,217,17963,10,9,0.9,20,16,0.80,30,24,0.8000,50,38,0.76
7,301.tiff,canine cutaneous mast cell tumor,chromatin_od,217,17963,10,6,0.6,20,15,0.75,30,25,0.8333,50,41,0.82
8,301.tiff,canine cutaneous mast cell tumor,od31,217,17963,10,6,0.6,20,16,0.80,30,25,0.8333,50,43,0.86
9,301.tiff,canine cutaneous mast cell tumor,od_falloff,217,17963,10,10,1.0,20,19,0.95,30,27,0.9000,50,43,0.86


## Table B -- precision per domain, per arm, at each budget

7 domains x 6 arms x 4 budgets = 168 rows. `precision_pooled = sum(tp_at_budget) /
sum(budget_delivered)` across that domain's 2 ROIs, for that arm -- algebraically identical to
the simple mean of the two ROIs' `precision_at_K` there, because the checks above guarantee
`budget_delivered == K` for both. The worst-ROI columns name the weaker of the two ROIs per
domain/arm/budget, so a bad cell can't hide behind the pooled number.

In [7]:
rows_b = []
for (domain, arm, k), g in RAW.groupby(['tumor_type', 'arm', 'budget']):
    tp_sum = int(g['tp_at_budget'].sum())
    delivered_sum = int(g['budget_delivered'].sum())
    worst = g.loc[g['precision_at_budget'].idxmin()]
    rows_b.append(dict(
        domain=domain, arm=arm, n_roi=len(g), K=int(k),
        tp_sum=tp_sum, delivered_sum=delivered_sum,
        precision_pooled=round(tp_sum / delivered_sum, 4) if delivered_sum else np.nan,
        precision_worst_roi=round(float(worst['precision_at_budget']), 4),
        worst_roi_file=str(worst['file_name']),
    ))

TABLE_B = pd.DataFrame(rows_b).sort_values(['domain', 'arm', 'K']).reset_index(drop=True)
TABLE_B.to_csv(OUT_BY_DOMAIN, index=False)
print(f'-> {OUT_BY_DOMAIN}  ({len(TABLE_B)} rows = 7 domains x {len(AXES)} arms x {len(BUDGETS)} budgets)')
TABLE_B

-> ../results/precision_at_k_14roi_gatedseed_chromatin_by_domain.csv  (168 rows = 7 domains x 6 arms x 4 budgets)


,domain,arm,n_roi,K,tp_sum,delivered_sum,precision_pooled,precision_worst_roi,worst_roi_file
0,canine cutaneous mast cell tumor,chromatin_od,2,10,15,20,0.7500,0.6000,301.tiff
1,canine cutaneous mast cell tumor,chromatin_od,2,20,33,40,0.8250,0.7500,301.tiff
2,canine cutaneous mast cell tumor,chromatin_od,2,30,52,60,0.8667,0.8333,301.tiff
3,canine cutaneous mast cell tumor,chromatin_od,2,50,84,100,0.8400,0.8200,301.tiff
4,canine cutaneous mast cell tumor,mask_od_mean,2,10,13,20,0.6500,0.4000,301.tiff
5,canine cutaneous mast cell tumor,mask_od_mean,2,20,30,40,0.7500,0.6500,301.tiff
6,canine cutaneous mast cell tumor,mask_od_mean,2,30,47,60,0.7833,0.7000,301.tiff
7,canine cutaneous mast cell tumor,mask_od_mean,2,50,76,100,0.7600,0.6800,301.tiff
8,canine cutaneous mast cell tumor,od31,2,10,16,20,0.8000,0.6000,301.tiff
9,canine cutaneous mast cell tumor,od31,2,20,35,40,0.8750,0.8000,301.tiff


## Closing summary

`precision_at_K` (Table A) and `precision_pooled` / `precision_worst_roi` (Table B), per arm,
are the deliverable. As in the reference notebook, `compare.evaluate_arms` computes
`recall_at_budget`, `full_list_recall` and `read_50..read_100` per arm as an unavoidable
byproduct of reusing that harness; those columns exist only in
`results/precision_at_k_14roi_gatedseed_chromatin_raw.csv`, for provenance, and are not part of this
analysis.

Three things D5 already said, worth restating here rather than assumed:

- **Single-seed.** This run answers D5's question at precision@K, on one click per ROI -- not
  the 5-seed sweep with the paired delta clustered at the ROI that D5 itself sets as the bar for
  changing the production ranker. Read Table B's numbers per arm as an observation about this
  run, not as a decision.
- **The axis set and the ROI set are not independent.** D5's 2026-09-08 amendment says outright
  that `od_contrast` was "selected from six axes scored on the same 14 ROIs it is then reported
  on." This notebook re-scores that same six-axis shortlist on that same 14-ROI set -- which is
  what D5's own standing constraint requires (any chromatin re-measurement must use
  `images/extra_valid`; there is no other approved set to test on), so the reuse is correct, not
  a flaw. But it means a win here for `chromatin_od` or `od_contrast` is not independent
  confirmation of the evidence that already favoured them on this set -- it is the same 14 ROIs
  asked a related question a second time.
- **`od_falloff`'s weak precision@K here is not a new finding.** D5 already recorded that, on
  `read_95` (a depth metric), `od_falloff` has the *best* median across these 14 ROIs of the
  axes compared there (1,955, versus `od_contrast`'s 3,074) but a worse worst-case, which is why
  `od_contrast` -- not `od_falloff` -- was the one to satisfy the 90%-floor shortlist rule.
  Precision@K is a different, top-of-list-sensitive metric than `read_95`, so this run's result
  is not a re-derivation of that split; it is a second observation, on a different metric,
  consistent with the same axis behaving inconsistently across summary statistics.

In [8]:
RAW.to_csv(OUT_RAW, index=False)
print(f'-> {OUT_RAW}  ({len(RAW)} rows; recall-family columns retained here only, for provenance)')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

-> ../results/precision_at_k_14roi_gatedseed_chromatin_raw.csv  (336 rows; recall-family columns retained here only, for provenance)

notebook ran in 430s
